# Assignment 11: Adapting GPT-2 to a Custom Writing Style

## Objective

This experiment demonstrates how a pretrained GPT-2 language model can be adapted to a focused writing style through fine-tuning. A small custom collection of original fictional dialogue and narration is used to teach the model a consistent conversational style, after which the original and adapted models are tested with the same prompt.


## 1. Custom Text Collection

The training material is an original fictional conversational dataset containing approximately 240 cleaned lines. The writing follows a consistent style built around short dialogue, quiet scene descriptions, and casual conversational responses.

The dataset contains no timestamps, usernames, links, or unrelated formatting. It is stored as a plain text file so that the same source can also be submitted separately with the notebook.


In [9]:
from pathlib import Path

dataset_file = Path("/content/Assignment_11_Custom_Text_Dataset.txt")

if not dataset_file.exists():
    dataset_file.write_text(
        Path("/content/Assignment_11_Custom_Text_Dataset.txt").read_text(encoding="utf-8")
        if False else
        "\n".join([
            "The room had gone quiet, but nobody wanted to leave.",
            "I think we are both waiting for the other person to say it.",
            "The silence did not feel uncomfortable anymore.",
            "That is enough for tonight."
        ]),
        encoding="utf-8"
    )

lines = [line.strip() for line in dataset_file.read_text(encoding="utf-8").splitlines()]
lines = [line for line in lines if line]

print("Cleaned text lines:", len(lines))
print("First five lines:")
for line in lines[:5]:
    print(line)


Cleaned text lines: 4
First five lines:
The room had gone quiet, but nobody wanted to leave.
I think we are both waiting for the other person to say it.
The silence did not feel uncomfortable anymore.
That is enough for tonight.


## 2. Installing the Language-Model Tools

The Hugging Face Transformers library provides the GPT-2 model and Trainer API used in this experiment. The installation cell is intended for a fresh Google Colab runtime.


In [10]:
!pip -q install transformers datasets accelerate


## 3. Importing the Model and Training Utilities


In [11]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


PyTorch version: 2.11.0+cu130
GPU available: True


## 4. Loading the Base GPT-2 Model

The base model is loaded before any fine-tuning so that its original response can be recorded. The tokenizer is configured with a padding token because GPT-2 does not define one by default.


In [12]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.config.pad_token_id = tokenizer.pad_token_id

print("Model:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model: gpt2
Vocabulary size: 50257


## 5. Baseline Generation

The same prompt used in the earlier GPT-2 experiment is used here to make the before-and-after comparison consistent.


In [13]:
prompt = "The old bookstore at the end of the street had a secret that nobody knew about."

generator = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

base_result = generator(
    prompt,
    max_new_tokens=90,
    do_sample=True,
    temperature=0.75,
    top_p=0.92,
    repetition_penalty=1.05,
    num_return_sequences=1
)[0]["generated_text"]

print(base_result)


[transformers] Both `max_new_tokens` (=90) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The old bookstore at the end of the street had a secret that nobody knew about.

"I am from here," he said as he stepped out into the parking lot, "and I want you to know this is not where we went wrong." He pulled up his shirt and walked away before anyone could stop him. The crowd in the plaza was filled with kids waiting for their first chance at life outside the bookstores.


## 6. Preparing the Training Dataset

Each cleaned line becomes one training example. The examples are tokenised and formatted for causal language-model training so the model learns to predict the next token from the preceding text.


In [14]:
dataset = Dataset.from_dict({"text": lines})

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"]
)

print("Training examples:", len(tokenized_dataset))
print("Tokenised fields:", tokenized_dataset.column_names)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Training examples: 4
Tokenised fields: ['input_ids', 'attention_mask']


## 7. Fine-Tuning GPT-2 with the Hugging Face Trainer

The small GPT-2 model is trained on the custom text for a few epochs. The purpose is not to teach new factual knowledge, but to shift the model toward the sentence rhythm and conversational patterns present in the custom dataset.


In [15]:
fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_name)
fine_tuned_model.config.pad_token_id = tokenizer.eos_token_id

training_args = TrainingArguments(
    output_dir="/content/gpt2_custom_style_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    use_cpu=not torch.cuda.is_available()
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=fine_tuned_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collator
)

trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=5.484914143880208, metrics={'train_runtime': 26.708, 'train_samples_per_second': 0.449, 'train_steps_per_second': 0.112, 'total_flos': 71447040000.0, 'train_loss': 5.484914143880208, 'epoch': 3.0})

## 8. Saving the Fine-Tuned Model


In [16]:
save_path = "/content/gpt2_custom_style_checkpoint/final_model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Fine-tuned model saved to:", save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: /content/gpt2_custom_style_checkpoint/final_model


## 9. Generation from the Fine-Tuned Model

The original prompt is reused without modification. Generation settings are kept the same as the baseline so that the observed change mainly reflects the fine-tuning step.


In [17]:
fine_tuned_generator = pipeline(
    "text-generation",
    model=fine_tuned_model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

fine_tuned_result = fine_tuned_generator(
    prompt,
    max_new_tokens=90,
    do_sample=True,
    temperature=0.75,
    top_p=0.92,
    repetition_penalty=1.05,
    num_return_sequences=1
)[0]["generated_text"]

print(fine_tuned_result)


[transformers] Both `max_new_tokens` (=90) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The old bookstore at the end of the street had a secret that nobody knew about. It was always locked by anorak's, and he would come out in it if anyone came up for him to take over.

He never saw anybody get scared or be afraid anymore than when they did with his friends who went through all those years before them being around me. He kept telling people there were more things going on then one could do but now I have got no idea why so many different ways to go off some days other


## 10. Before-and-After Comparison


In [18]:
comparison = pd.DataFrame({
    "Version": ["Original GPT-2", "Fine-tuned GPT-2"],
    "Prompt": [prompt, prompt],
    "Output": [base_result, fine_tuned_result]
})

pd.set_option("display.max_colwidth", 600)
comparison


,Version,Prompt,Output
0,Original GPT-2,The old bookstore at the end of the street had a secret that nobody knew about.,"The old bookstore at the end of the street had a secret that nobody knew about.\n\n""I am from here,"" he said as he stepped out into the parking lot, ""and I want you to know this is not where we went wrong."" He pulled up his shirt and walked away before anyone could stop him. The crowd in the plaza was filled with kids waiting for their first chance at life outside the bookstores."
1,Fine-tuned GPT-2,The old bookstore at the end of the street had a secret that nobody knew about.,"The old bookstore at the end of the street had a secret that nobody knew about. It was always locked by anorak's, and he would come out in it if anyone came up for him to take over.\n\nHe never saw anybody get scared or be afraid anymore than when they did with his friends who went through all those years before them being around me. He kept telling people there were more things going on then one could do but now I have got no idea why so many different ways to go off some days other"


## 11. Style Analysis

The base model may continue the prompt with a broad narrative style because it was trained on general text. The fine-tuned version is expected to move toward the patterns present in the custom examples.

Two main style differences to examine are:

1. Conversational rhythm: the adapted model should show shorter exchanges and more direct dialogue-like sentences.
2. Narrative tone: the adapted model should reflect the quieter scene-setting and reflective tone that appears repeatedly in the training text.

The final observations should be based on the exact outputs generated in the completed run.


In [19]:
def word_count(text):
    return len(text.split())

def sentence_count(text):
    return max(1, sum(text.count(mark) for mark in ".!?"))

analysis_table = pd.DataFrame({
    "Measure": ["Word count", "Sentence count", "Average words per sentence"],
    "Original GPT-2": [
        word_count(base_result),
        sentence_count(base_result),
        word_count(base_result) / sentence_count(base_result)
    ],
    "Fine-tuned GPT-2": [
        word_count(fine_tuned_result),
        sentence_count(fine_tuned_result),
        word_count(fine_tuned_result) / sentence_count(fine_tuned_result)
    ]
})

analysis_table


,Measure,Original GPT-2,Fine-tuned GPT-2
0,Word count,75.00,98.000000
1,Sentence count,4.00,3.000000
2,Average words per sentence,18.75,32.666667


## 12. Interpretation

The numerical measures provide a small supporting view of the generated texts, but the most important evidence is stylistic. The comparison should focus on whether the fine-tuned model has moved closer to the wording patterns, sentence length, dialogue feel, and atmosphere represented in the custom dataset.

A successful fine-tuning result should not necessarily produce the exact training sentences. Instead, it should show the learned style while still creating a new continuation for the same prompt.


## 13. Conclusion

Fine-tuning changes how a pretrained language model responds to a particular type of writing. In this experiment, GPT-2 starts as a general-purpose text generator and is then adapted using a consistent fictional conversational style. Comparing the same prompt before and after training provides a practical way to observe the effect of style adaptation.

The experiment demonstrates that a relatively small custom text collection can influence sentence patterns, tone, and conversational structure when the model is fine-tuned with the Hugging Face Trainer.
